# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import xmltodict
import requests
import time
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

os.chdir(downloads)

In [2]:
# Read metadata

metadata_folder = downloads + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(metadata) # 7397 rows

# Find only >= 2024 using run ID from metadata
metadata["Collection_Date"] = pd.to_numeric(metadata["Collection_Date"], errors='coerce')
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats

print(metadata_new) # 5750 rows




              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR28752446        WGS      146.11   93605195  PRJNA1102327   
1     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
2     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
3     SRR28752449        WGS      146.61   59363690  PRJNA1102327   
4     SRR28752450        WGS      251.31  119232569  PRJNA1102327   
...           ...        ...         ...        ...           ...   
7392  SRR32633103        WGS      148.75  108436208  PRJNA1102327   
7393  SRR32633104        WGS      148.77   90195365  PRJNA1102327   
7394  SRR32633105        WGS      148.64   89418970  PRJNA1102327   
7395  SRR32633106        WGS      148.58   84745077  PRJNA1102327   
7396  SRR32633107        WGS      148.50  114370864  PRJNA1102327   

         BioSample BioSampleModel     Bytes Center Name  Collection_Date  ...  \
0     SAMN41019184          Viral  30074178   USDA-NVSL             2024  ...   
1     SAM

C:\Users\maksiaevai\AppData\Local\Temp\2\ipykernel_11324\2612779725.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats


In [4]:
display(metadata_new)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,ReleaseDate,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-008354-001-original,SRP503016,H5N1,NaN,SRS21079812,False,NaN
1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-005-original,SRP503016,H5N1,NaN,SRS21079811,False,NaN
2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-004-original,SRP503016,H5N1,NaN,SRS21079810,False,NaN
3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-003-original,SRP503016,H5N1,NaN,SRS21079809,False,NaN
4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-002-original,SRP503016,H5N1,NaN,SRS21079808,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7392,SRR32633103,WGS,148.75,108436208,PRJNA1102327,SAMN47290837,Viral,40656883,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:13,1,25-005512-003,SRP503016,NaN,"MILK, BULK TANK",SRS24304291,False,NaN
7393,SRR32633104,WGS,148.77,90195365,PRJNA1102327,SAMN47290836,Viral,33878271,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:13,1,25-005512-002,SRP503016,NaN,"MILK, BULK TANK",SRS24304283,False,NaN
7394,SRR32633105,WGS,148.64,89418970,PRJNA1102327,SAMN47290835,Viral,33592911,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:15,1,25-005512-001,SRP503016,NaN,"MILK, BULK TANK",SRS24304286,False,NaN
7395,SRR32633106,WGS,148.58,84745077,PRJNA1102327,SAMN47290826,Viral,32103811,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:17,1,25-005501-002,SRP503016,NaN,"MILK, BULK TANK",SRS24304281,False,NaN


### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use output.tsv

In [6]:
# Get genotype from genoflu
os.chdir(temp_files)
output_tsv = pd.read_csv("output.tsv", delimiter="\t")

b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})

metadata_new = metadata_new.merge(b313_and_d11_only, on="Run")

# print(metadata_new) # 6267 rows -- are some genotypes duplicating entries?

metadata_uniques = metadata_new.drop_duplicates(subset="Run")

print(metadata_uniques) # 5160 rows???

              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR28752446        WGS      146.11   93605195  PRJNA1102327   
4     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
8     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
12    SRR28752449        WGS      146.61   59363690  PRJNA1102327   
16    SRR28752450        WGS      251.31  119232569  PRJNA1102327   
...           ...        ...         ...        ...           ...   
8476  SRR32633103        WGS      148.75  108436208  PRJNA1102327   
8477  SRR32633104        WGS      148.77   90195365  PRJNA1102327   
8478  SRR32633105        WGS      148.64   89418970  PRJNA1102327   
8479  SRR32633106        WGS      148.58   84745077  PRJNA1102327   
8480  SRR32633107        WGS      148.50  114370864  PRJNA1102327   

         BioSample BioSampleModel     Bytes Center Name  Collection_Date  ...  \
0     SAMN41019184          Viral  30074178   USDA-NVSL             2024  ...   
4     SAM

In [ ]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_new, animals_ref) # Get host type
metadata_new["years"] = metadata_new["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

c:\Users\maksiaevai\Documents\Avian_Flu\utils.py:207: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata["Host_Type"] = animal_types
C:\Users\maksiaevai\AppData\Local\Temp\2\ipykernel_23080\2768212524.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["years"] = metadata_new["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date


              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR28752446        WGS      146.11   93605195  PRJNA1102327   
1     SRR28752446        WGS      146.11   93605195  PRJNA1102327   
2     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
3     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
4     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
...           ...        ...         ...        ...           ...   
6852  SRR32633103        WGS      148.75  108436208  PRJNA1102327   
6853  SRR32633104        WGS      148.77   90195365  PRJNA1102327   
6854  SRR32633105        WGS      148.64   89418970  PRJNA1102327   
6855  SRR32633106        WGS      148.58   84745077  PRJNA1102327   
6856  SRR32633107        WGS      148.50  114370864  PRJNA1102327   

         BioSample BioSampleModel     Bytes Center Name  Collection_Date  ...  \
0     SAMN41019184          Viral  30074178   USDA-NVSL             2024  ...   
1     SAM

In [ ]:
# Get geolocation from genbank_mapping.tsv

os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2])

# map to metadata

genbank_mapping["isolate"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[3])

metadata_genbank = metadata_new.merge(genbank_mapping, on="isolate", how="left")

print(metadata_genbank)

               Run Assay Type  AvgSpotLen      Bases    BioProject  \
0      SRR28752446        WGS      146.11   93605195  PRJNA1102327   
1      SRR28752446        WGS      146.11   93605195  PRJNA1102327   
2      SRR28752447        WGS      241.29   86080323  PRJNA1102327   
3      SRR28752447        WGS      241.29   86080323  PRJNA1102327   
4      SRR28752448        WGS      250.30   75035343  PRJNA1102327   
...            ...        ...         ...        ...           ...   
10229  SRR32633103        WGS      148.75  108436208  PRJNA1102327   
10230  SRR32633104        WGS      148.77   90195365  PRJNA1102327   
10231  SRR32633105        WGS      148.64   89418970  PRJNA1102327   
10232  SRR32633106        WGS      148.58   84745077  PRJNA1102327   
10233  SRR32633107        WGS      148.50  114370864  PRJNA1102327   

          BioSample BioSampleModel     Bytes Center Name  Collection_Date  \
0      SAMN41019184          Viral  30074178   USDA-NVSL             2024   
1    

In [ ]:
# Get collection date from GenBank eutils 

def search_collection_date(biosample):

    print(biosample)
    
    # Avoid spamming the server
    time.sleep(2)
    
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

    # Get Biosample ID from search_url
    output = requests.get(search_url)
    xml = output.content
    root = ET.fromstring(xml)
    sample_id = root.find("./IdList/Id").text

    biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
    
    # Get Nucleotide ID from biosample_url
    output = requests.get(biosample_url)
    xml = output.content
    root = ET.fromstring(xml)
    query_key = root.find(".//QueryKey").text
    web_env = root.find(".//WebEnv").text

    nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

    output = requests.get(nucleotide_url) 
    xml = output.content
    root = ET.fromstring(xml)

    # Grab collection date at the end of the sub name
    collection_date = root.find(".//SubName").text.split("|")[-1]

    return collection_date
    
metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(search_collection_date)

nan


TypeError: can only concatenate str (not "float") to str

In [ ]:
# Make names

# names = ">A/" + metadata_new["Host"] + "/" + metadata_new["geo_loc_name"] + "/" + metadata_new["isolate"] + "/" + years + "|H5N1|" + metadata_new["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata_new["Host_Type"] + "|" + metadata_new["Genotype"]
names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"] + "|H5N1|" + metadata_genbank["Collection_Date_Specific"] + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

display(metadata_genbank)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Genotype,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Name
0,SRR28752470,WGS,147.45,81957492,PRJNA1102327,SAMN41019216,Viral,26941955,USDA-NVSL,2024,...,B3.13,SRR28752471_HA_cns.fa,Consensus_SRR28752471_HA_cns_threshold_0.5_qua...,SRR28752471,HA,PQ011563.1,4,A/Cattle/Michigan/24-009027-002-v/2024,Michigan,>A/Cattle/Michigan/24-009027-002-v/2024|H5N1|2...
1,SRR28752470,WGS,147.45,81957492,PRJNA1102327,SAMN41019216,Viral,26941955,USDA-NVSL,2024,...,B3.13,SRR28752471_MP_cns.fa,Consensus_SRR28752471_MP_cns_threshold_0.5_qua...,SRR28752471,MP,PQ011566.1,7,A/Cattle/Michigan/24-009027-002-v/2024,Michigan,>A/Cattle/Michigan/24-009027-002-v/2024|H5N1|2...
2,SRR28752470,WGS,147.45,81957492,PRJNA1102327,SAMN41019216,Viral,26941955,USDA-NVSL,2024,...,B3.13,SRR28752471_NA_cns.fa,Consensus_SRR28752471_NA_cns_threshold_0.5_qua...,SRR28752471,NaN,PQ011565.1,6,A/Cattle/Michigan/24-009027-002-v/2024,Michigan,>A/Cattle/Michigan/24-009027-002-v/2024|H5N1|2...
3,SRR28752470,WGS,147.45,81957492,PRJNA1102327,SAMN41019216,Viral,26941955,USDA-NVSL,2024,...,B3.13,SRR28752471_NP_cns.fa,Consensus_SRR28752471_NP_cns_threshold_0.5_qua...,SRR28752471,NP,PQ011564.1,5,A/Cattle/Michigan/24-009027-002-v/2024,Michigan,>A/Cattle/Michigan/24-009027-002-v/2024|H5N1|2...
4,SRR28752470,WGS,147.45,81957492,PRJNA1102327,SAMN41019216,Viral,26941955,USDA-NVSL,2024,...,B3.13,SRR28752471_NS_cns.fa,Consensus_SRR28752471_NS_cns_threshold_0.5_qua...,SRR28752471,NS,PQ011567.1,8,A/Cattle/Michigan/24-009027-002-v/2024,Michigan,>A/Cattle/Michigan/24-009027-002-v/2024|H5N1|2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4451,SRR31167894,WGS,147.42,65957232,PRJNA980729,SAMN44493025,Viral,20955917,USDA-NVSL,2024,...,D1.1,SRR31167894_NP_cns.fa,Consensus_SRR31167894_NP_cns_threshold_0.5_qua...,SRR31167894,NP,PQ664442.1,5,A/chicken/WA/24-030039-001/2024,WA,>A/CHICKEN/WA/24-030039-001/2024|H5N1|2024|avi...
4452,SRR31167894,WGS,147.42,65957232,PRJNA980729,SAMN44493025,Viral,20955917,USDA-NVSL,2024,...,D1.1,SRR31167894_NS_cns.fa,Consensus_SRR31167894_NS_cns_threshold_0.5_qua...,SRR31167894,NS,PQ664445.1,8,A/chicken/WA/24-030039-001/2024,WA,>A/CHICKEN/WA/24-030039-001/2024|H5N1|2024|avi...
4453,SRR31167894,WGS,147.42,65957232,PRJNA980729,SAMN44493025,Viral,20955917,USDA-NVSL,2024,...,D1.1,SRR31167894_PA_cns.fa,Consensus_SRR31167894_PA_cns_threshold_0.5_qua...,SRR31167894,PA,PQ664440.1,3,A/chicken/WA/24-030039-001/2024,WA,>A/CHICKEN/WA/24-030039-001/2024|H5N1|2024|avi...
4454,SRR31167894,WGS,147.42,65957232,PRJNA980729,SAMN44493025,Viral,20955917,USDA-NVSL,2024,...,D1.1,SRR31167894_PB1_cns.fa,Consensus_SRR31167894_PB1_cns_threshold_0.5_qu...,SRR31167894,PB1,PQ664439.1,2,A/chicken/WA/24-030039-001/2024,WA,>A/CHICKEN/WA/24-030039-001/2024|H5N1|2024|avi...


In [ ]:
# Make fasta files

fasta_folder = downloads + "avian-influenza/fasta/"

os.chdir(fasta_folder)

pairs = []
fasta_files = {}

for genotype in ["B3.13", "D1.1"]:
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = []

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            fasta_files[pair].append(header)
                            fasta_files[pair].append(sequence)
                f.close()

In [ ]:
# print(fasta_files[list(fasta_files.keys())[0]])

print(len(fasta_files["B3.13_PB1"]))

8880


In [ ]:
# Create fasta files 
os.chdir(complete_files)
for pair in fasta_files.keys():
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        item = fasta_files[pair]
        name = str(item[0])
        # print(item[1])
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

In [ ]:
def search_collection_date(biosample):
    
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

    # Get Biosample ID from search_url
    output = requests.get(search_url)
    xml = output.content
    root = ET.fromstring(xml)
    sample_id = root.find("./IdList/Id").text

    biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
    
    # Get Nucleotide ID from biosample_url
    output = requests.get(biosample_url)
    xml = output.content
    root = ET.fromstring(xml)
    query_key = root.find(".//QueryKey").text
    web_env = root.find(".//WebEnv").text

    nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

    output = requests.get(nucleotide_url) 
    xml = output.content
    root = ET.fromstring(xml)

    # Grab collection date at the end of the sub name
    collection_date = root.find(".//SubName").text.split("|")[-1]

    # Avoid spamming the server
    time.sleep(3)
    
    return collection_date


In [ ]:
example = search_collection_date("SAMN41019216")
print(example)



24-Mar-2024


In [ ]:

# unique_animals_all = sort_animals_anderson(metadata_new)

# # Flatten unique_animals
# every_unique_animal = []
# for animal in unique_animals_all:
#     every_unique_animal.append(animal)

# print(every_unique_animal)

# unique_animals_set = list(set(every_unique_animal))
# # animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# # animals_df["other"] = unique_animals_set # to sort

# os.chdir(downloads)

# animals_ref = pd.read_csv("animals_ref.csv")


# # If animal not in ref1, put in ref2

# common_animals = []
# # Check if animals in unique_animals_set are in ref1
# for animal in unique_animals_set:
#     for col in animals_ref.columns:
#         if animal in animals_ref[col].values and type(animal) == str:
#             common_animals.append(animal)

# print(common_animals)
# print(len(common_animals))

# different_animals = []
# for animal in unique_animals_set:
#     if animal not in common_animals:
#         different_animals.append(animal)

# print(different_animals)

# # Add to dataframe
# animals_df = animals_ref
# # Make different_animals same length as dataframe, if shorter
# if len(different_animals) < len(animals_df):
#     number_of_times_to_add_nan = len(animals_df) - len(different_animals)
#     for i in range(number_of_times_to_add_nan):
#         different_animals.append(float('nan'))
# # If longer, deal with that later

# animals_df["new"] = (different_animals)

# print(animals_df)

# animals_df.to_csv("animals_ref_to_sort.csv")
